# Focus-StyleGAN 工业缺陷增广 · Kaggle 训练 Notebook

毕业设计项目：面向 MVTec AD 的双分支 GAN 缺陷图像增广模型训练。
训练完成后把 `checkpoints/` 拉回本地，即可衔接 datapipe 管线 B（合成数据质量过滤）。

## 使用前必做（Kaggle 设置）

1. **加速器**：右侧面板 Settings → Accelerator → **GPU T4**（16GB 显存，本 notebook 默认 batch_size=16 适配）。
2. **联网**：Settings → Internet → **On**（需要手机验证的账号；用于 pip 安装缺失依赖和下载 VGG/Inception 预训练权重）。
3. **挂载两个输入数据集**（右侧面板 Add Input）：
   - **项目代码**：把本地项目 `Industrial-Defect-Multimodal-Pipeline` 打包 zip，上传为 Kaggle Dataset；
   - **MVTec AD 数据集**：Kaggle 上有公开镜像（搜索 mvtec ad），也可自行上传官网下载的原始数据
     （https://www.mvtec.com/company/research/datasets/mvtec-ad ，免费注册）。
     公开镜像非官方发布，挂载后第 3 节会自动校验目录结构，若校验失败请换用官网原版。

## Kaggle 限额（以官方说明为准，可能调整）

- GPU 配额约 **30 小时/周**，单次会话最长约 **12 小时**。
- 因此策略是**每次会话训一个品类**：本 notebook 默认 `bottle` 30 epoch，跑完保存版本（Save Version）后，
  把 `checkpoints/` 上传为 Dataset 挂到下一次会话即可断点续训（见最后一节）。

执行顺序：从上到下 Run All 即可。

## 1. 参数区（按需修改后 Run All）

In [ ]:
# ============ 可调参数 ============
CATEGORY = 'bottle'    # 本次训练的品类；None 表示全品类（不推荐：单次会话时长不够）
EPOCHS = 30          # 试水建议 20-30；T4 上单品类 30 epoch 约数小时（估算，以实际为准）
BATCH_SIZE = 16      # T4 16GB 安全值；P100 16GB 同；若确认显存富余可改 32
RESUME_CHECKPOINT = None  # 断点续训：填挂载的 checkpoint 路径，
                          # 如 '/kaggle/input/<ckpt数据集>/checkpoint_epoch_30.pth'
SEED = 42            # 与 core/integrated_config.yaml 默认一致，保证可复现
print(f'品类={CATEGORY}, epochs={EPOCHS}, batch_size={BATCH_SIZE}, 续训={RESUME_CHECKPOINT}')

## 2. 环境检查（GPU / torch / 显存）

In [ ]:
import torch, subprocess, sys
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__)
assert torch.cuda.is_available(), '未检测到 GPU！请先在右侧 Settings → Accelerator 选择 GPU T4'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name}（显存 {mem:.1f} GB）')
if mem < 12:
    print('⚠️ 显存 <12GB，请把参数区 BATCH_SIZE 降到 8')
device = torch.device('cuda')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                      '--format=csv'], capture_output=True, text=True).stdout)

## 3. 定位代码与数据集 + 安装缺失依赖

在 `/kaggle/input` **任意深度**自动查找项目代码（含 `core/integrated_config.yaml`）
和 MVTec AD（含 `品类/train/good/*.png` 且 PNG 最多的候选，自动排除代码包里的 demo 数据），
定位失败时会打印 `/kaggle/input` 的实际结构便于排查。
找到后把代码复制到可写的 `/kaggle/working/project`。

In [ ]:
import shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PROJECT = WORK / 'project'

def find_code_root():
    """在 /kaggle/input 任意深度定位项目根（含 core/integrated_config.yaml）。"""
    for p in sorted(INPUT.rglob('integrated_config.yaml')):
        if p.is_file() and p.parent.name == 'core':
            return p.parent.parent
    return None

def find_mvtec_root(exclude=None):
    """在 /kaggle/input 任意深度定位 MVTec AD 根（含 品类/train/good/*.png）。

    按 train/good 下 PNG 数量取最多的候选，并排除项目代码目录——
    代码包里 datasets/demo_mvtec_synthetic 结构相同，不能误当真实数据。
    """
    best, best_n = None, 0
    for p in sorted(INPUT.rglob('good')):
        if not (p.is_dir() and p.parent.name == 'train'):
            continue
        if exclude is not None and exclude in p.parents:
            continue
        n = len(list(p.glob('*.png')))
        if n > best_n:
            best, best_n = p.parent.parent.parent, n
    return best

def print_input_tree(max_depth=4, max_lines=60):
    """诊断用：打印 /kaggle/input 的实际结构（定位失败时看清真相）。"""
    print('--- /kaggle/input 实际结构（诊断输出）---')
    lines = 0
    for p in sorted(INPUT.rglob('*')):
        depth = len(p.relative_to(INPUT).parts)
        if depth <= max_depth:
            print('  ' * depth + p.name + ('/' if p.is_dir() else ''))
            lines += 1
            if lines >= max_lines:
                print('  ...（截断）')
                break

code_src = find_code_root()
data_src = find_mvtec_root(exclude=code_src)

if code_src is None or data_src is None:
    print_input_tree()
    msg = chr(10).join([
        f'定位失败: code_src={code_src}, data_src={data_src}。',
        '请对照上面打印的实际结构排查：',
        '1) 右侧 Add Input 是否真的挂载成功（面板显示不代表挂载到本会话，',
        '   可尝试 Restart & Clear Outputs 后 Run All）；',
        '2) 若仍失败，请把上面的诊断输出发出来对照。'])
    raise AssertionError(msg)
print('代码源:', code_src)
print('数据源:', data_src)

# 复制代码到可写目录（input 只读，checkpoint 要写入项目内）
shutil.copytree(code_src, PROJECT, dirs_exist_ok=True)
print('代码已复制到:', PROJECT)

# 数据集结构校验
cats = sorted(p for p in data_src.iterdir() if p.is_dir())
n_train = sum(len(list((c/'train'/'good').glob('*.png'))) for c in cats if (c/'train'/'good').is_dir())
print(f'品类数: {len(cats)}，训练集正常样本共 {n_train} 张')
if CATEGORY:
    assert (data_src/CATEGORY/'train'/'good').is_dir(), f'品类不存在: {CATEGORY}，可用: {[c.name for c in cats]}'
    print(f'本次品类 {CATEGORY}: train/good {len(list((data_src/CATEGORY/"train"/"good").glob("*.png")))} 张')

# 安装 Kaggle 镜像缺失的依赖（lpips / optuna；其余 CV 依赖镜像自带）
!pip install -q lpips optuna
print('依赖安装完成')

## 4. 生成 Kaggle 专用配置副本

不改动原始 `core/integrated_config.yaml`，而是生成副本并覆盖：
数据集路径、batch_size、num_workers（Kaggle 仅 4 核 CPU）、epochs、checkpoint 目录。

In [ ]:
import yaml

cfg_path = PROJECT / 'core' / 'integrated_config.yaml'
cfg = yaml.safe_load(open(cfg_path, encoding='utf-8'))
cfg['data']['dataset_path'] = str(data_src)
cfg['data']['batch_size'] = BATCH_SIZE
cfg['data']['num_workers'] = 2          # Kaggle 实例 4 核 CPU
cfg['training']['epochs'] = EPOCHS
cfg['training']['seed'] = SEED
cfg['training']['checkpoint_dir'] = str(PROJECT / 'checkpoints')

KG_CONFIG = WORK / 'kaggle_config.yaml'
yaml.safe_dump(cfg, open(KG_CONFIG, 'w', encoding='utf-8'), allow_unicode=True)
print('Kaggle 配置已生成:', KG_CONFIG)
print(f'dataset={cfg["data"]["dataset_path"]}, batch={cfg["data"]["batch_size"]}, '
      f'epochs={cfg["training"]["epochs"]}, fid_interval={cfg["training"]["fid_interval"]}')

## 5. 启动训练

装配逻辑与本地 `cloud/train_cloud.py` 一致（复刻 `create_trainer_from_config`，
仅多传 `category` 参数），输出实时打印 loss / FID 进度，同时写入 `train_kaggle.log`。

**首次运行会下载 VGG / InceptionV3 / LPIPS 预训练权重（几百 MB），需保持 Internet On。**

In [ ]:
import sys, logging, time
sys.path.insert(0, str(PROJECT))

from model.utils.config import Config
from model.utils.seed import set_seed
from model.data.loader import MVTecADDataset
from model.models.focus_stylegan import FocusStyleGAN
from model.training.trainer import FocusStyleGANTrainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(),
                              logging.FileHandler(PROJECT/'train_kaggle.log', encoding='utf-8')],
                    force=True)
logger = logging.getLogger('kaggle_train')

config = Config(str(KG_CONFIG))
set_seed(config.training.get('seed', SEED))

dataset = MVTecADDataset(root_path=config.data.dataset_path,
                         image_size=config.data.image_size,
                         augment=config.data.augmentations.use,
                         category=CATEGORY)   # 数据集类原生支持；None = 全品类
model = FocusStyleGAN(config=config.model).to(device)
trainer = FocusStyleGANTrainer(model=model, dataset=dataset, config=config,
                               device=device, logger=logger)

t0 = time.time()
trainer.train(n_epochs=EPOCHS, resume_from=RESUME_CHECKPOINT)
hours = (time.time() - t0) / 3600

print('=' * 60)
print(f'训练完成，耗时 {hours:.2f} 小时')
print(f'checkpoint 目录: {trainer.checkpoint_dir}')
if trainer.best_fid != float('inf'):
    print(f'最佳 FID: {trainer.best_fid:.2f}（best_model.pth）')
else:
    print('最佳 FID: 未计算（epoch 数不足 fid_interval）')

# 训练历史落盘：内核重启后，第 6 节画图可直接从磁盘恢复
import json as _json
history_payload = {'category': CATEGORY, 'epochs': EPOCHS, 'hours': round(hours, 3),
                   'best_fid': None if trainer.best_fid == float('inf') else trainer.best_fid,
                   'history': trainer.train_history}
with open(PROJECT / 'train_history.json', 'w', encoding='utf-8') as f:
    _json.dump(history_payload, f, ensure_ascii=False, indent=2)
print('训练历史已保存:', PROJECT / 'train_history.json')

## 6. 训练曲线（loss / FID）

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# 内核重启兼容：优先内存，其次落盘 JSON，最后解析 train_kaggle.log
PROJECT = Path('/kaggle/working/project')
hist = None
if 'trainer' in globals():
    hist = trainer.train_history
    source = 'memory (this session)'
elif (PROJECT / 'train_history.json').exists():
    payload = json.load(open(PROJECT / 'train_history.json', encoding='utf-8'))
    hist = payload['history']
    source = f"train_history.json (category={payload.get('category')})"
else:
    # 兼容旧版 notebook 已完成的训练：从日志逐行解析 G/D 损失与 FID
    log_path = PROJECT / 'train_kaggle.log'
    assert log_path.exists(), ('no training history found: run section 5 first, '
                               'or make sure project/train_kaggle.log exists')
    hist = {'g_loss': [], 'd_loss': [], 'fid': []}
    # 日志为中文格式「训练 - G损失: X, D损失: Y」「FID: Z」
    import re
    for line in open(log_path, encoding='utf-8'):
        m = re.search(r'训练 - G\S*: ([-\d.]+), D\S*: ([-\d.]+)', line)
        if m:
            hist['g_loss'].append(float(m.group(1)))
            hist['d_loss'].append(float(m.group(2)))
            continue
        m = re.search(r'FID: ([-\d.]+)', line)
        if m:
            hist['fid'].append(float(m.group(1)))
    source = 'train_kaggle.log (parsed)'
assert hist['g_loss'], 'training history is empty'
print('history source:', source)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist['g_loss'], label='G loss')
axes[0].plot(hist['d_loss'], label='D loss')
axes[0].set_title('Adversarial Loss'); axes[0].set_xlabel('epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

if hist['fid']:
    axes[1].plot(hist['fid'], marker='o', color='darkred')
    axes[1].set_title('FID (lower is better, computed every fid_interval epochs)')
else:
    axes[1].text(0.5, 0.5, 'epochs < fid_interval: no FID computed',
                 ha='center', va='center')
axes[1].set_xlabel('eval point'); axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(PROJECT / 'training_curves.png', dpi=120)
plt.show()
print('saved:', PROJECT / 'training_curves.png')

## 7. 生成效果 inline 验收

加载 `best_model.pth`（无 best 则用 final），对测试集正常图生成伪异常：
左列原图，右列生成结果（不同随机种子 + 缺陷强度）。
生成模型输出范围为 [-1, 1]，显示时映射回 [0, 1]。

In [ ]:
# 内核重启守卫：本单元格依赖第 1-4 节的变量；重启后先重跑第 1-4 节（秒级，不会重新训练）
missing = [v for v in ('config', 'data_src', 'PROJECT') if v not in globals()]
assert not missing, (f'missing variables {missing}: kernel was restarted. '
                     'Re-run sections 1-4 first (fast, no retraining), then this cell.')

import numpy as np
from PIL import Image
from torchvision import transforms as T
from model.augmentation.augmentor import Augmentor

ckpt = PROJECT/'checkpoints'/'best_model.pth'
if not ckpt.exists():
    ckpt = PROJECT/'checkpoints'/'final_model.pth'
print('加载:', ckpt)

aug = Augmentor(config=config, device=device, logger=logger)
aug.load_model(ckpt)

size = config.data.image_size
tf = T.Compose([T.Resize((size, size)), T.ToTensor(),
                T.Normalize([0.5]*3, [0.5]*3)])   # 与训练预处理一致 → [-1,1]
good_imgs = sorted((data_src/(CATEGORY or cats[0].name)/'test'/'good').glob('*.png'))[:3]
assert good_imgs, '测试集 good 样本为空'

n = len(good_imgs)
fig, axes = plt.subplots(n, 3, figsize=(10, 3.2*n))
for i, p in enumerate(good_imgs):
    real = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
    fake1 = aug.generate_pseudo_anomaly(real, defect_intensity=1.0, rng_seed=SEED+i)
    fake2 = aug.generate_pseudo_anomaly(real, defect_intensity=1.5, rng_seed=SEED+100+i)
    def show(ax, t, title):
        img = (t.detach().squeeze().cpu().permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy()
        ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
    show(axes[i, 0], real,  f'original {p.name}')
    show(axes[i, 1], fake1, 'generated (intensity 1.0)')
    show(axes[i, 2], fake2, 'generated (intensity 1.5)')
fig.tight_layout()
fig.savefig(PROJECT / 'generated_samples.png', dpi=120)
plt.show()
print('样例图已保存:', PROJECT / 'generated_samples.png')

## 8.5 训练产出自动备份到 Kaggle Dataset（防丢失）

**背景**：Kaggle 的 Quick Save 只保存代码，不保存 `/kaggle/working` 产出；
会话一关，未备份的 checkpoint 就永久丢失。本单元格在训练结束后自动把
checkpoints 推送为一个 Kaggle Dataset（续训时挂载它即可）。

**一次性前置设置**：notebook 编辑页 → Add-ons → **Secrets** → 添加两个条目：
`KAGGLE_USERNAME`（你的用户名）和 `KAGGLE_KEY`
（在 https://www.kaggle.com/settings → API → Create New Token 下载的
kaggle.json 里能查到 key）。

没配置 Secrets 时本单元格只打印提示，不会报错中断。

In [ ]:
# 训练产出自动备份到 Kaggle Dataset
import json as _json
from pathlib import Path as _Path

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _user = _secrets.get_secret('KAGGLE_USERNAME')
    _key = _secrets.get_secret('KAGGLE_KEY')
except Exception as _e:
    _user = None
    print('未配置 Kaggle Secrets，跳过自动备份。')
    print('请立即手动保存产出：Save Version 只能保存代码，')
    print('产出文件需要用 Output 面板的 "New Dataset" 按钮转存！')
    print('（一劳永逸：Add-ons → Secrets 添加 KAGGLE_USERNAME / KAGGLE_KEY 后重跑本单元格）')

if _user:
    import os, subprocess
    os.environ['KAGGLE_USERNAME'] = _user
    os.environ['KAGGLE_KEY'] = _key

    tag = CATEGORY or 'all'
    slug = f'fsgan-ckpt-{tag}'
    backup_dir = WORK / 'backup' / slug
    backup_dir.mkdir(parents=True, exist_ok=True)

    # 打包 checkpoint + 训练历史 + 曲线 + 样例 + 日志
    shutil.copytree(PROJECT / 'checkpoints', backup_dir / 'checkpoints', dirs_exist_ok=True)
    for extra in ['train_history.json', 'training_curves.png',
                  'generated_samples.png', 'train_kaggle.log']:
        p = PROJECT / extra
        if p.exists():
            shutil.copy(p, backup_dir / extra)

    # dataset 元数据（Kaggle 要求）
    meta = {'title': slug, 'id': f'{_user}/{slug}', 'licenses': [{'name': 'CC0-1.0'}]}
    (backup_dir / 'dataset-metadata.json').write_text(
        _json.dumps(meta), encoding='utf-8')

    # 已存在则更新版本，不存在则创建
    check = subprocess.run(['kaggle', 'datasets', 'status', f'{_user}/{slug}'],
                           capture_output=True, text=True)
    if check.returncode == 0:
        r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(backup_dir),
                            '-m', f'update after {EPOCHS} epochs (best_fid={trainer.best_fid})',
                            '--dir-mode', 'zip'],
                           capture_output=True, text=True)
    else:
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(backup_dir),
                            '--dir-mode', 'zip'],
                           capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode == 0:
        print(f'备份成功: https://www.kaggle.com/datasets/{_user}/{slug}')
        print('续训时挂载该数据集，RESUME_CHECKPOINT 指向其中的 checkpoint 文件即可')
    else:
        print('备份失败（不影响训练结果本身），请手动用 Output 面板 "New Dataset" 转存')

## 8. 打包产出 + 断点续训说明

**本次产出**（都在右侧 Output 面板可下载）：
- `checkpoints_<品类>.zip` —— 全部 checkpoint（含 best_model.pth）
- `project/training_curves.png`、`project/generated_samples.png`、`project/train_kaggle.log`

**断点续训（Kaggle 单次会话时长有限的标准做法）**：
1. 本次结束后 Save Version；
2. 把 Output 里的 checkpoint zip 下载后**上传为 Kaggle Dataset**（如 `fsgan-ckpt-bottle`）；
3. 下次会话 Add Input 挂载它，参数区设置
   `RESUME_CHECKPOINT = '/kaggle/input/fsgan-ckpt-bottle/checkpoint_epoch_30.pth'`，Run All 即可续训。

**回本地衔接管线**：把 `best_model.pth` 放入本地项目根 `checkpoints/` 目录，然后：
```bash
python core/integrated_augmentor.py --mode single --input <good.png> --variations 50
python -m datapipe.pipeline_b --input <生成图目录>
```

In [ ]:
tag = CATEGORY or 'all'
zip_path = shutil.make_archive(str(WORK / f'checkpoints_{tag}'), 'zip',
                               PROJECT / 'checkpoints')
print('打包完成:', zip_path)
print('\n===== 本次训练摘要（可誊入 evaluation/results.md）=====')
print(f'平台: Kaggle GPU ({torch.cuda.get_device_name(0)})')
print(f'品类: {tag}, epochs: {EPOCHS}, batch_size: {BATCH_SIZE}, seed: {SEED}')
print(f'耗时: {hours:.2f} 小时')
if trainer.best_fid != float('inf'):
    print(f'最佳 FID: {trainer.best_fid:.2f}')
print(f'checkpoint: {ckpt.name}')
print('备注: 上述数字为真实训练产出，誊写时请附本 notebook 的 Kaggle 链接作为出处')